In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('../data/cookie_cats.csv')
print("Shape:", df.shape)
df.head(10)

Shape: (90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True
5,540,gate_40,187,True,True
6,1066,gate_30,0,False,False
7,1444,gate_40,2,False,False
8,1574,gate_40,108,True,True
9,1587,gate_40,153,True,False


In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.2 MB


In [6]:
df.describe()

,userid,sum_gamerounds
count,9.018900e+04,90189.000000
mean,4.998412e+06,51.872457
std,2.883286e+06,195.050858
min,1.160000e+02,0.000000
25%,2.512230e+06,5.000000
50%,4.995815e+06,16.000000
75%,7.496452e+06,51.000000
max,9.999861e+06,49854.000000


In [7]:
print("=== NULL VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print(f"Số dòng trùng lặp: {df.duplicated().sum()}")

print("\n=== UNIQUE VALUES PER COLUMN ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

=== NULL VALUES ===
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

=== DUPLICATE ROWS ===
Số dòng trùng lặp: 0

=== UNIQUE VALUES PER COLUMN ===
userid: 90189 unique values
version: 2 unique values
sum_gamerounds: 942 unique values
retention_1: 2 unique values
retention_7: 2 unique values


In [8]:
print("=== PHÂN BỔ NHÓM A/B ===")
print(df['version'].value_counts())
print(f"\nTỷ lệ: {df['version'].value_counts(normalize=True).round(3)}")

=== PHÂN BỔ NHÓM A/B ===
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64

Tỷ lệ: version
gate_40    0.504
gate_30    0.496
Name: proportion, dtype: float64


In [9]:
# Xác định và loại bỏ outlier
outlier_threshold = df['sum_gamerounds'].max()
print(f"Giá trị outlier: {outlier_threshold}")

df_clean = df[df['sum_gamerounds'] < outlier_threshold]
print(f"Số dòng trước khi loại: {len(df)}")
print(f"Số dòng sau khi loại: {len(df_clean)}")

df_clean['sum_gamerounds'].describe()

Giá trị outlier: 49854
Số dòng trước khi loại: 90189
Số dòng sau khi loại: 90188


count    90188.000000
mean        51.320253
std        102.682719
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max       2961.000000
Name: sum_gamerounds, dtype: float64

In [10]:
df_clean.to_csv('../data/cookie_cats_clean.csv', index=False)
print("Đã lưu file thành công")

Đã lưu file thành công


In [11]:
import sqlite3

conn = sqlite3.connect('../data/cookie_cats.db')
df_clean.to_sql('game_data', conn, if_exists='replace', index=False)

print("Đã tạo database thành công")

test_query = "SELECT * FROM game_data LIMIT 5"
pd.read_sql(test_query, conn)

Đã tạo database thành công


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,0,0
1,337,gate_30,38,1,0
2,377,gate_40,165,1,0
3,483,gate_40,1,0,0
4,488,gate_40,179,1,1


In [12]:
q1 = """
SELECT version, COUNT(*) as total_users
FROM game_data
GROUP BY version
"""
pd.read_sql(q1, conn)

,version,total_users
0,gate_30,44699
1,gate_40,45489


In [13]:
q2 = """
SELECT 
    version,
    AVG(sum_gamerounds) as avg_rounds,
    MAX(sum_gamerounds) as max_rounds,
    MIN(sum_gamerounds) as min_rounds
FROM game_data
GROUP BY version
"""
pd.read_sql(q2, conn)

,version,avg_rounds,max_rounds,min_rounds
0,gate_30,51.342111,2961,0
1,gate_40,51.298776,2640,0


In [14]:
q3 = """
SELECT 
    version,
    AVG(CASE WHEN retention_1 = 1 THEN 1.0 ELSE 0 END) as retention_1_rate,
    AVG(CASE WHEN retention_7 = 1 THEN 1.0 ELSE 0 END) as retention_7_rate
FROM game_data
GROUP BY version
"""
pd.read_sql(q3, conn)

,version,retention_1_rate,retention_7_rate
0,gate_30,0.448198,0.190183
1,gate_40,0.442283,0.182000


In [15]:
q4 = """
SELECT 
    userid, version, sum_gamerounds,
    PERCENT_RANK() OVER (ORDER BY sum_gamerounds) as percentile_rank
FROM game_data
ORDER BY sum_gamerounds DESC
LIMIT 10
"""
pd.read_sql(q4, conn)

,userid,version,sum_gamerounds,percentile_rank
0,871500,gate_30,2961,1.000000
1,3271615,gate_40,2640,0.999989
2,4832608,gate_30,2438,0.999978
3,5346171,gate_40,2294,0.999967
4,5133952,gate_30,2251,0.999956
5,9640085,gate_30,2156,0.999945
6,4090246,gate_40,2124,0.999933
7,9791599,gate_40,2063,0.999922
8,725080,gate_40,2015,0.999911
9,69927,gate_30,1906,0.999900


In [16]:
q5 = """
SELECT 
    userid, 
    COUNT(*) as count_occurrence
FROM game_data
GROUP BY userid
HAVING COUNT(*) > 1
"""
result_q5 = pd.read_sql(q5, conn)
print(f"Số userid bị trùng: {len(result_q5)}")
result_q5

Số userid bị trùng: 0


,userid,count_occurrence


In [17]:
from scipy.stats import chi2_contingency

# Tạo bảng chéo (contingency table) cho retention_7
contingency_table = pd.crosstab(df_clean['version'], df_clean['retention_7'])
print("Bảng tần suất:")
print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nKết luận: p-value ({p_value:.4f}) < {alpha} → Chênh lệch CÓ Ý NGHĨA thống kê")
else:
    print(f"\nKết luận: p-value ({p_value:.4f}) >= {alpha} → Chênh lệch KHÔNG đủ bằng chứng để khẳng định là thật, có thể do ngẫu nhiên")

Bảng tần suất:
retention_7  False  True 
version                  
gate_30      36198   8501
gate_40      37210   8279

Chi-square statistic: 9.9153
P-value: 0.0016

Kết luận: p-value (0.0016) < 0.05 → Chênh lệch CÓ Ý NGHĨA thống kê


In [18]:
contingency_table_r1 = pd.crosstab(df_clean['version'], df_clean['retention_1'])
print("Bảng tần suất (retention_1):")
print(contingency_table_r1)

chi2_r1, p_value_r1, dof_r1, expected_r1 = chi2_contingency(contingency_table_r1)
print(f"\nChi-square statistic: {chi2_r1:.4f}")
print(f"P-value: {p_value_r1:.4f}")

if p_value_r1 < alpha:
    print(f"\nKết luận: Chênh lệch retention_1 CÓ Ý NGHĨA thống kê")
else:
    print(f"\nKết luận: Chênh lệch retention_1 KHÔNG đủ bằng chứng để khẳng định là thật")

Bảng tần suất (retention_1):
retention_1  False  True 
version                  
gate_30      24665  20034
gate_40      25370  20119

Chi-square statistic: 3.1698
P-value: 0.0750

Kết luận: Chênh lệch retention_1 KHÔNG đủ bằng chứng để khẳng định là thật


m